# AI Node in the Notebook DAG — UI/UX Design

**Surface:** the Jute notebook DAG view (`crates/spur-notebook/jute-notebook/src/ui/dag/`) — `DagView` canvas + `DagNode` cards + `DagInspector` side panel.
**Audience:** notebook authors wiring a reactive data graph who now want an *AI node* (a `spur`-kernelspec cell that runs its body as a prompt against the configured agent, with upstream Arrow ports as context).
**Goal:** make an AI node legible *in situ* next to code nodes, and surface the things only an AI node has — the agent it runs on, manual-vs-live mode, token usage, cache hits, and the "no agent configured" degraded state.

## Direction — reuse the locked visual system (Tech Utility)
The DAG UI already has a brand: neutral white cards, gray hairlines, **status is the only colour** (3px left rail + header dot: `fresh` emerald · `stale` amber · `running` blue · `failed` red · `upstream-failed` red-300 · `never-run` gray). The card stays neutral so a wall of nodes reads *topology first, status second*. I extend that system rather than invent a new one — the closest of the five Open Design directions is **Tech Utility (Datadog/GitHub)**, which the existing UI already is.

## The one real design decision — how to mark an AI node without breaking "status is the only colour"
An AI node can be fresh/stale/running/failed just like a code node, so node-**kind** is *orthogonal* to status. Options:
- **A. Monochrome glyph only** — a slate `✦ AI` tag, zero new colour. Safest, but easy to miss in a dense graph.
- **B. (recommended) One restrained violet accent, used ≤2× per node** — a `✦ agent` tag + the `LIVE` pill. Violet is reserved for *AI-kind* and never fills the card, so status colours stay authoritative. This is the Tech-Utility "tinted status pill" idiom applied to a new dimension.
- **C. Distinct card surface (tinted fill / dashed border)** — most obvious, but it overrides the topology-first neutral-surface rule and reads as slop. Rejected.

I take **B**, with the prompt body rendered as quoted roman text (not mono code) as a second, colour-free signal that this node speaks natural language.

## New affordances unique to AI nodes (mapped to the merged backend)
| UI element | Backend source |
|---|---|
| `✦ agent` tag | `AgentConfig` selected by `ai_backend_from_config` (brain default → first → none) |
| `LIVE` / `manual` mode pill | `ai_live` flag (manual-run by default; live = auto on source-push) |
| token meter (`▸ 1.2k → 380`) | `AiUsage { input, output }` from the prompt turn |
| `cached` chip | input-hash cache hit in `NotebookCellRunner` (blake3 of prompt + port versions) |
| context-port tokens `↓` | `PortContext` rendered from upstream Arrow ports |
| `text` produced-port glyph | the single text port an AI node writes |
| **`needs agent`** degraded state | `NullAiBackend` → `AiError::Init` when no agent is configured |

## Plan
1. **Canvas in situ** — AI node sitting among code nodes, edges intact.
2. **Anatomy** — one enlarged AI node with callouts to each backend field.
3. **State gallery** — manual · live · running(streaming) · cached · stale · needs-agent · failed.
4. **Inspector** — the `DagInspector` panel adapted for an AI node (agent picker, mode toggle, context ports, prompt editor, usage, Run / Run-live).

Artifact renders below as a single self-contained `text/html` cell.

In [3]:
# open-design artifact — AI node in the notebook DAG
from IPython.display import HTML

HTML(r"""
<div id="ad-root">
<style>
#ad-root{--bg:#f8fafc;--surface:#fff;--fg:#111827;--muted:#6b7280;--faint:#9ca3af;
--border:#e5e7eb;--border2:#d1d5db;--fresh:#10b981;--stale:#f59e0b;--running:#3b82f6;
--failed:#ef4444;--upfail:#fca5a5;--neverrun:#d1d5db;--ai:#7c3aed;--ai-bg:#f5f3ff;--ai-bd:#ddd6fe;
--sans:-apple-system,BlinkMacSystemFont,'Inter','Segoe UI',system-ui,sans-serif;
--mono:'JetBrains Mono','IBM Plex Mono',ui-monospace,Menlo,monospace;
background:var(--bg);color:var(--fg);font-family:var(--sans);font-size:14px;
line-height:1.5;padding:28px 30px 40px;border-radius:8px;}
#ad-root *{box-sizing:border-box;}
#ad-root h1{font-size:21px;font-weight:700;letter-spacing:-0.02em;margin:0;}
#ad-root .sub{color:var(--muted);font-size:13px;margin-top:4px;max-width:760px;}
#ad-root .kicker{font-family:var(--mono);font-size:10.5px;font-weight:600;letter-spacing:0.08em;
text-transform:uppercase;color:var(--faint);margin:34px 0 14px;border-top:1px solid var(--border);padding-top:14px;}
#ad-root .legend{display:flex;flex-wrap:wrap;gap:14px;font-family:var(--mono);font-size:11px;color:var(--muted);margin-top:10px;}
#ad-root .lg{display:inline-flex;align-items:center;gap:6px;}
#ad-root .sw{width:10px;height:10px;border-radius:50%;}
/* ---- node card (mirrors DagNode.tsx: 224px, 3px rail, dot, mono preview, port row) ---- */
#ad-root .node{width:224px;background:var(--surface);border:1px solid var(--border);border-radius:6px;
box-shadow:0 1px 2px rgba(0,0,0,.05);display:flex;overflow:hidden;}
#ad-root .rail{width:3px;flex:0 0 auto;}
#ad-root .nb{flex:1;min-width:0;padding:8px 10px;}
#ad-root .nh{display:flex;align-items:center;gap:8px;}
#ad-root .dot{width:8px;height:8px;border-radius:50%;flex:0 0 auto;}
#ad-root .nl{font-size:12.5px;font-weight:600;color:var(--fg);white-space:nowrap;overflow:hidden;text-overflow:ellipsis;flex:1;min-width:0;}
#ad-root .nid{font-family:var(--mono);font-size:10px;color:var(--faint);flex:0 0 auto;}
#ad-root .pv{margin-top:5px;font-family:var(--mono);font-size:10.5px;color:var(--muted);white-space:nowrap;overflow:hidden;text-overflow:ellipsis;}
#ad-root .pr{margin-top:8px;display:flex;align-items:center;justify-content:space-between;gap:8px;font-family:var(--mono);font-size:10px;color:var(--muted);}
#ad-root .pr .l,#ad-root .pr .r{display:flex;align-items:center;gap:6px;min-width:0;overflow:hidden;}
#ad-root .arrow{color:var(--faint);flex:0 0 auto;}
#ad-root .ver{color:var(--faint);}
#ad-root .stl{color:#b45309;}
#ad-root .stl-d{width:4px;height:4px;border-radius:50%;background:var(--stale);flex:0 0 auto;}
/* status colors */
.s-fresh{background:var(--fresh);} .s-stale{background:var(--stale);} .s-running{background:var(--running);}
.s-failed{background:var(--failed);} .s-upfail{background:var(--upfail);} .s-neverrun{background:var(--neverrun);}
#ad-root .dot.hollow{background:#fff;border:1.5px solid var(--border2);}
#ad-root .dot.run{animation:adpulse 1.1s ease-in-out infinite;}
@keyframes adpulse{0%,100%{opacity:1;}50%{opacity:.35;}}
/* AI-specific (kind marker — violet, used at most twice per card) */
#ad-root .aisub{margin-top:6px;display:flex;align-items:center;gap:6px;flex-wrap:wrap;}
#ad-root .tag-ai{display:inline-flex;align-items:center;gap:3px;font-family:var(--mono);font-size:9.5px;
font-weight:600;color:var(--ai);background:var(--ai-bg);border:1px solid var(--ai-bd);border-radius:4px;padding:1px 5px;}
#ad-root .pill{font-family:var(--mono);font-size:9px;border-radius:4px;padding:1px 5px;border:1px solid var(--border2);color:var(--muted);background:#fff;}
#ad-root .pill.live{color:#fff;background:var(--ai);border-color:var(--ai);}
#ad-root .prompt{margin-top:6px;font-size:11px;line-height:1.36;color:#374151;
display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical;overflow:hidden;}
#ad-root .prompt .q{color:var(--ai);font-weight:700;margin-right:2px;}
#ad-root .meta{margin-top:7px;display:flex;align-items:center;gap:7px;font-family:var(--mono);font-size:9.5px;color:var(--faint);}
#ad-root .chip{font-family:var(--mono);font-size:9px;border-radius:4px;padding:0 5px;border:1px solid var(--border);color:var(--muted);background:#fafafa;}
#ad-root .chip.warn{color:#92400e;background:#fffbeb;border-color:#fde68a;}
#ad-root .gT{font-family:var(--mono);font-size:8px;border:1px solid var(--border2);border-radius:2px;padding:0 2px;color:var(--muted);}
#ad-root .um{height:4px;border-radius:2px;background:#ede9fe;overflow:hidden;display:flex;}
#ad-root .um .in{background:#c4b5fd;} #ad-root .um .out{background:var(--ai);}
/* canvas */
#ad-root .canvas{position:relative;height:344px;background:
linear-gradient(var(--border) 1px,transparent 1px),linear-gradient(90deg,var(--border) 1px,transparent 1px);
background-size:22px 22px;background-color:#fff;border:1px solid var(--border);border-radius:8px;overflow:hidden;}
#ad-root .canvas .node{position:absolute;}
#ad-root svg.edges{position:absolute;inset:0;width:100%;height:100%;pointer-events:none;}
/* anatomy + galleries */
#ad-root .row{display:flex;gap:30px;align-items:flex-start;flex-wrap:wrap;}
#ad-root .callouts{flex:1;min-width:280px;display:grid;gap:9px;}
#ad-root .co{display:flex;gap:10px;align-items:flex-start;font-size:12.5px;}
#ad-root .co b{font-family:var(--mono);font-size:10px;color:#fff;background:var(--ai);border-radius:50%;width:17px;height:17px;
display:inline-flex;align-items:center;justify-content:center;flex:0 0 auto;margin-top:1px;}
#ad-root .co .t{color:var(--fg);} #ad-root .co .t span{color:var(--muted);}
#ad-root .co code{font-family:var(--mono);font-size:11px;background:#f1f5f9;border-radius:3px;padding:0 4px;color:#334155;}
#ad-root .num{position:absolute;font-family:var(--mono);font-size:9.5px;color:#fff;background:var(--ai);border-radius:50%;
width:16px;height:16px;display:flex;align-items:center;justify-content:center;}
#ad-root .gallery{display:grid;grid-template-columns:repeat(auto-fill,minmax(236px,1fr));gap:18px;}
#ad-root .cell .cap{font-family:var(--mono);font-size:10.5px;color:var(--fg);margin-top:8px;font-weight:600;}
#ad-root .cell .cd{font-size:11.5px;color:var(--muted);margin-top:2px;line-height:1.4;}
/* inspector */
#ad-root .insp{width:320px;background:#fff;border:1px solid var(--border);border-radius:8px;padding:16px;flex:0 0 auto;}
#ad-root .insp .ttl{font-family:var(--mono);font-size:10.5px;font-weight:600;letter-spacing:.04em;text-transform:uppercase;color:var(--faint);}
#ad-root .insp h3{margin:4px 0 0;font-size:16px;font-weight:600;}
#ad-root .insp .iid{font-family:var(--mono);font-size:11px;color:var(--muted);margin-top:2px;}
#ad-root .insp .badges{margin-top:9px;display:flex;gap:6px;flex-wrap:wrap;}
#ad-root .insp .sect{margin-top:16px;}
#ad-root .insp .sh{font-family:var(--mono);font-size:10px;font-weight:600;letter-spacing:.04em;text-transform:uppercase;color:var(--faint);margin-bottom:7px;}
#ad-root .insp .pl{display:flex;justify-content:space-between;align-items:center;font-size:12.5px;padding:2px 0;}
#ad-root .insp .pl .pn{font-weight:500;color:#1f2937;}
#ad-root .vb{font-family:var(--mono);font-size:10.5px;border-radius:4px;padding:1px 6px;background:#f0f9ff;color:#0369a1;}
#ad-root .vb.bump{background:#fef3c7;color:#92400e;}
#ad-root .vb.em{background:#ecfdf5;color:#047857;}
#ad-root .sel{display:flex;align-items:center;justify-content:space-between;border:1px solid var(--border2);border-radius:6px;
padding:7px 10px;font-size:12.5px;font-family:var(--mono);color:#1f2937;}
#ad-root .seg{display:inline-flex;border:1px solid var(--border2);border-radius:6px;overflow:hidden;font-family:var(--mono);font-size:11px;}
#ad-root .seg span{padding:5px 12px;color:var(--muted);}
#ad-root .seg span.on{background:var(--ai);color:#fff;}
#ad-root .pbox{border:1px solid var(--border2);border-radius:6px;padding:9px 11px;font-size:12.5px;color:#374151;line-height:1.45;background:#fff;}
#ad-root .usage{display:flex;justify-content:space-between;font-family:var(--mono);font-size:11px;color:var(--muted);margin-bottom:6px;}
#ad-root .btns{margin-top:16px;display:flex;gap:8px;}
#ad-root .btn{flex:1;display:inline-flex;align-items:center;justify-content:center;gap:6px;border:1px solid var(--border2);
border-radius:6px;padding:8px;font-size:12.5px;font-weight:500;color:#1f2937;background:#fff;}
#ad-root .btn.prim{background:var(--ai);color:#fff;border-color:var(--ai);}
#ad-root .btn.dis{opacity:.5;}
#ad-root .note{font-size:12px;color:var(--muted);margin-top:10px;max-width:640px;}
#ad-root table.map{border-collapse:collapse;width:100%;font-size:12.5px;margin-top:4px;}
#ad-root table.map th{text-align:left;font-family:var(--mono);font-size:10px;text-transform:uppercase;letter-spacing:.04em;
color:var(--faint);font-weight:600;border-bottom:1px solid var(--border);padding:6px 10px;}
#ad-root table.map td{border-bottom:1px solid var(--border);padding:7px 10px;vertical-align:top;}
#ad-root table.map code{font-family:var(--mono);font-size:11px;background:#f5f3ff;color:#6d28d9;border-radius:3px;padding:0 4px;}
</style>

<h1>The AI node, in the notebook DAG</h1>
<div class="sub">A <code style="font-family:var(--mono);font-size:12px">spur</code>-kernelspec cell that runs its body as a prompt against the configured agent, with upstream Arrow ports as context. It lives on the same canvas as code nodes and obeys the same status language — it just carries four things only an AI node has: an <b>agent</b>, a <b>mode</b>, <b>token usage</b>, and a <b>needs-agent</b> fallback.</div>

<div class="legend">
  <span class="lg"><span class="sw s-fresh"></span>fresh</span>
  <span class="lg"><span class="sw s-stale"></span>stale</span>
  <span class="lg"><span class="sw s-running"></span>running</span>
  <span class="lg"><span class="sw s-failed"></span>failed</span>
  <span class="lg"><span class="sw s-upfail"></span>upstream-failed</span>
  <span class="lg"><span class="sw s-neverrun"></span>never-run</span>
  <span class="lg"><span class="tag-ai">&#10022; AI</span>= node kind, not status</span>
</div>

<div class="kicker">1 · On the canvas — an AI node among code nodes</div>
<div class="canvas">
  <svg class="edges" viewBox="0 0 1040 344" preserveAspectRatio="none">
    <path d="M264,86 C330,86 340,150 400,150" fill="none" stroke="#cbd5e1" stroke-width="1.5"/>
    <path d="M264,250 C330,250 340,196 400,196" fill="none" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 3"/>
    <path d="M624,176 C690,176 720,150 772,150" fill="none" stroke="#cbd5e1" stroke-width="1.5"/>
  </svg>

  <div class="node" style="left:40px;top:40px;">
    <div class="rail s-fresh"></div><div class="nb">
      <div class="nh"><span class="dot s-fresh"></span><span class="nl">sales</span><span class="nid">a1</span></div>
      <div class="pv">df = pl.read_parquet(src)</div>
      <div class="pr"><div class="l"><span class="arrow">&#8595;</span><span>weekly.csv</span></div><div class="r"><span>sales <span class="ver">v3</span></span><span class="arrow">&#8593;</span></div></div>
    </div>
  </div>

  <div class="node" style="left:40px;top:204px;">
    <div class="rail s-fresh"></div><div class="nb">
      <div class="nh"><span class="dot s-fresh"></span><span class="nl">targets</span><span class="nid">a2</span></div>
      <div class="pv">targets = load_targets()</div>
      <div class="pr"><div class="l"><span class="arrow">&#8595;</span><span class="stl"><span class="stl-d"></span>quota.json</span></div><div class="r"><span>targets <span class="ver">v1</span></span><span class="arrow">&#8593;</span></div></div>
    </div>
  </div>

  <!-- the AI node, center stage -->
  <div class="node" style="left:400px;top:108px;">
    <div class="rail s-stale"></div><div class="nb">
      <div class="nh"><span class="dot s-stale"></span><span class="nl">summary</span><span class="nid">a3</span></div>
      <div class="aisub"><span class="tag-ai">&#10022; claude-opus</span><span class="pill">manual</span></div>
      <div class="prompt"><span class="q">&#8220;</span>Summarise this week's sales against targets in 3 bullets; call out the biggest miss.</div>
      <div class="pr"><div class="l"><span class="arrow">&#8595;</span><span>sales <span class="ver">v3</span></span><span class="stl"><span class="stl-d"></span>targets</span></div><div class="r"><span class="gT">T</span><span>summary</span><span class="arrow">&#8593;</span></div></div>
      <div class="meta"><span>&#9656; 1.2k &#8594; 380 tok</span><span class="chip">cached</span></div>
    </div>
  </div>

  <div class="node" style="left:772px;top:114px;">
    <div class="rail s-neverrun"></div><div class="nb">
      <div class="nh"><span class="dot hollow"></span><span class="nl">report</span><span class="nid">a4</span></div>
      <div class="pv">render_md(summary)</div>
      <div class="pr"><div class="l"><span class="arrow">&#8595;</span><span>summary <span class="ver">v?</span></span></div><div class="r"><span class="gray-300" style="color:var(--faint)">sink</span></div></div>
    </div>
  </div>
</div>
<div class="note">The AI node reads <b>topology first</b>: same neutral card, same amber rail telling you it is <b>stale</b> because <code style="font-family:var(--mono)">targets</code> bumped to a version it has not run against (amber dashed edge). The violet <span class="tag-ai" style="font-size:9px">&#10022; agent</span> tag and prompt text are the only things that say &ldquo;this one thinks.&rdquo;</div>

<div class="kicker">2 · Anatomy — every mark maps to the merged backend</div>
<div class="row">
  <div style="position:relative;flex:0 0 auto;">
    <div class="node" style="width:300px;">
      <div class="rail s-fresh"></div><div class="nb" style="padding:11px 13px;">
        <div class="nh"><span class="dot s-fresh"></span><span class="nl" style="font-size:14px">summary</span><span class="nid">a3</span></div>
        <div class="aisub"><span class="tag-ai">&#10022; claude-opus</span><span class="pill live">&#9679; LIVE</span></div>
        <div class="prompt" style="font-size:12px"><span class="q">&#8220;</span>Summarise this week's sales against targets in 3 bullets; call out the biggest miss.</div>
        <div class="pr" style="font-size:10.5px"><div class="l"><span class="arrow">&#8595;</span><span>sales <span class="ver">v3</span></span><span>targets <span class="ver">v1</span></span></div><div class="r"><span class="gT">T</span><span>summary</span><span class="arrow">&#8593;</span></div></div>
        <div class="meta" style="font-size:10px"><span>&#9656; 1.2k &#8594; 380 tok</span><span class="chip">cached</span></div>
      </div>
    </div>
    <span class="num" style="left:-9px;top:30px;">1</span>
    <span class="num" style="left:60px;top:58px;">2</span>
    <span class="num" style="left:165px;top:58px;">3</span>
    <span class="num" style="left:120px;top:96px;">4</span>
    <span class="num" style="right:6px;top:128px;">5</span>
    <span class="num" style="left:18px;top:150px;">6</span>
  </div>
  <div class="callouts">
    <div class="co"><b>1</b><div class="t">3px rail + dot = run <b>status</b> (fresh/stale/running/failed). Orthogonal to AI — an AI node goes stale exactly like code. <span>&larr; existing status language, untouched.</span></div></div>
    <div class="co"><b>2</b><div class="t"><code>&#10022; agent</code> — which <code>AgentConfig</code> the run binds to, chosen by <code>ai_backend_from_config</code> (brain default &rarr; first &rarr; none).</div></div>
    <div class="co"><b>3</b><div class="t"><code>manual</code> / <code>&#9679; LIVE</code> pill = the <code>ai_live</code> flag. Manual by default; LIVE auto-runs on source-push cascades.</div></div>
    <div class="co"><b>4</b><div class="t">Prompt as quoted roman text, not mono code — a colour-free signal the body is natural language, drained from the agent via <code>AcpAgentBackend</code>.</div></div>
    <div class="co"><b>5</b><div class="t"><code>T</code> text-port glyph — the one text port an AI node writes; downstream code consumes it like any Arrow port.</div></div>
    <div class="co"><b>6</b><div class="t"><code>&#9656; in &#8594; out tok</code> from <code>AiUsage</code>; <code>cached</code> chip = input-hash hit (blake3 of prompt + consumed port versions) — no agent call, no spend.</div></div>
  </div>
</div>

<div class="kicker">3 · State gallery</div>
<div class="gallery">
  <div class="cell">
    <div class="node"><div class="rail s-fresh"></div><div class="nb"><div class="nh"><span class="dot s-fresh"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="tag-ai">&#10022; opus</span><span class="pill">manual</span></div><div class="prompt"><span class="q">&#8220;</span>Summarise sales vs targets…</div><div class="meta"><span>&#9656; 1.2k &#8594; 380</span></div></div></div>
    <div class="cap">manual · fresh</div><div class="cd">Default. Ran on demand; output current. Won't re-run when upstream changes.</div>
  </div>
  <div class="cell">
    <div class="node"><div class="rail s-fresh"></div><div class="nb"><div class="nh"><span class="dot s-fresh"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="tag-ai">&#10022; opus</span><span class="pill live">&#9679; LIVE</span></div><div class="prompt"><span class="q">&#8220;</span>Summarise sales vs targets…</div><div class="meta"><span>&#9656; 1.2k &#8594; 380</span></div></div></div>
    <div class="cap">live</div><div class="cd">Re-runs automatically when a consumed port bumps (source-push cascade). Spend follows edits.</div>
  </div>
  <div class="cell">
    <div class="node"><div class="rail s-running"></div><div class="nb"><div class="nh"><span class="dot s-running run"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="tag-ai">&#10022; opus</span><span class="pill">manual</span></div><div class="prompt"><span class="q">&#8220;</span>Summarise sales vs t&#9611;</div><div class="meta"><span>&#9656; streaming&hellip;</span></div></div></div>
    <div class="cap">running · streaming</div><div class="cd">Blue pulse. Prompt turn draining; tokens stream into the produced text port live.</div>
  </div>
  <div class="cell">
    <div class="node"><div class="rail s-stale"></div><div class="nb"><div class="nh"><span class="dot s-stale"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="tag-ai">&#10022; opus</span><span class="pill">manual</span></div><div class="prompt"><span class="q">&#8220;</span>Summarise sales vs targets…</div><div class="pr"><div class="l"><span class="arrow">&#8595;</span><span class="stl"><span class="stl-d"></span>targets v1&#8594;v2</span></div><div class="r"><span class="gT">T</span><span class="arrow">&#8593;</span></div></div></div></div>
    <div class="cap">stale</div><div class="cd">A consumed port advanced past the version it last ran on. Output may be wrong; re-run to refresh.</div>
  </div>
  <div class="cell">
    <div class="node"><div class="rail s-neverrun"></div><div class="nb"><div class="nh"><span class="dot hollow"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="chip warn">&#9888; needs agent</span></div><div class="prompt" style="color:var(--faint)"><span class="q" style="color:var(--faint)">&#8220;</span>Summarise sales vs targets…</div><div class="meta"><span style="color:var(--faint)">run disabled · configure an agent</span></div></div></div>
    <div class="cap">needs agent</div><div class="cd"><code style="font-family:var(--mono);background:#f5f3ff;color:#6d28d9;border-radius:3px;padding:0 3px">NullAiBackend</code> &rarr; <code style="font-family:var(--mono);background:#f5f3ff;color:#6d28d9;border-radius:3px;padding:0 3px">AiError::Init</code>. No <code style="font-family:var(--mono)">.spur/config.toml</code> agent. Not runnable until set.</div>
  </div>
  <div class="cell">
    <div class="node"><div class="rail s-failed"></div><div class="nb"><div class="nh"><span class="dot s-failed"></span><span class="nl">summary</span><span class="nid">a3</span></div><div class="aisub"><span class="tag-ai">&#10022; opus</span><span class="pill">manual</span></div><div class="prompt"><span class="q">&#8220;</span>Summarise sales vs targets…</div><div class="meta"><span style="color:#b91c1c">&#9888; agent error · timeout</span></div></div></div>
    <div class="cap">failed</div><div class="cd">The prompt turn errored (transport/timeout). Red status; reason surfaces in the inspector.</div>
  </div>
</div>

<div class="kicker">4 · Inspector — the side panel, adapted for an AI node</div>
<div class="row">
  <div class="insp">
    <div class="ttl">Selected node</div>
    <h3>summary</h3>
    <div class="iid">a3</div>
    <div class="badges"><span class="pill" style="background:#fffbeb;border-color:#fde68a;color:#92400e">stale</span><span class="tag-ai">&#10022; claude-opus</span></div>

    <div class="sect"><div class="sh">Agent</div>
      <div class="sel"><span>claude-opus</span><span style="color:var(--faint)">&#9662;</span></div>
    </div>
    <div class="sect"><div class="sh">Mode</div>
      <div class="seg"><span class="on">manual</span><span>live</span></div>
    </div>
    <div class="sect"><div class="sh">Context · consumes</div>
      <div class="pl"><span class="pn">sales</span><span class="vb">v3</span></div>
      <div class="pl"><span class="pn">targets</span><span class="vb bump">v1 &rarr; v2</span></div>
    </div>
    <div class="sect"><div class="sh">Produces</div>
      <div class="pl"><span class="pn">summary <span class="gT">T</span></span><span class="vb em">v5</span></div>
    </div>
    <div class="sect"><div class="sh">Prompt</div>
      <div class="pbox">Summarise this week's sales against targets in 3 bullets; call out the biggest miss.</div>
    </div>
    <div class="sect"><div class="sh">Usage · last run</div>
      <div class="usage"><span>1,200 in &middot; 380 out</span><span>cached</span></div>
      <div class="um"><div class="in" style="width:76%"></div><div class="out" style="width:24%"></div></div>
    </div>
    <div class="btns"><span class="btn prim">&#9654; Run node</span><span class="btn">Run downstream</span></div>
  </div>

  <div style="flex:1;min-width:280px;">
    <div style="font-size:13px;color:var(--fg);font-weight:600;margin-bottom:8px;">What changed vs the code-cell inspector</div>
    <div class="note" style="margin-top:0;">The panel keeps its existing skeleton — <b>Consumes</b> / <b>Produces</b> version lists, <b>Run node</b> / <b>Run downstream</b> — and adds only what is AI-specific:</div>
    <div class="callouts" style="margin-top:12px;">
      <div class="co"><b>A</b><div class="t"><b>Agent</b> selector — pick the <code>AgentConfig</code> for this node, defaulting to the resolved brain/first agent. Empty &rarr; the <code>needs agent</code> state.</div></div>
      <div class="co"><b>M</b><div class="t"><b>Mode</b> toggle — <code>manual</code> &harr; <code>live</code> writes the <code>ai_live</code> flag. The single control that opts a node into source-push auto-runs.</div></div>
      <div class="co"><b>P</b><div class="t"><b>Prompt</b> replaces the code editor: a plain-text box (the cell body), not a syntax-highlit editor.</div></div>
      <div class="co"><b>U</b><div class="t"><b>Usage</b> meter from <code>AiUsage</code> — input vs output tokens, with the <code>cached</code> flag when the run was an input-hash hit.</div></div>
    </div>
    <div class="note">Run button binds to the existing <code style="font-family:var(--mono)">notebook_run_cell</code> path, which already wraps the runner with the AI backend (<code style="font-family:var(--mono)">notebook_run_context</code>). <b>Live mode</b> depends on follow-up <code style="font-family:var(--mono)">bd-1bpb</code> — wiring the AI backend into <code style="font-family:var(--mono)">spawn_reactive_engine</code> so source-push cascades run AI nodes too. Until then the <code>live</code> toggle is shown <i>disabled with a tooltip</i> rather than hidden.</div>
  </div>
</div>

<div class="kicker">5 · Integration map — UI &rarr; backend</div>
<table class="map">
  <thead><tr><th>UI mark</th><th>Drives / reads</th><th>Where it lives</th></tr></thead>
  <tbody>
    <tr><td><span class="tag-ai">&#10022; agent</span> tag &amp; Agent selector</td><td><code>AgentConfig</code> resolution</td><td><code>ai_backend_from_config</code> · <code>build_agent_connection</code> (run_context.rs)</td></tr>
    <tr><td><code>manual</code>/<code>LIVE</code> pill &amp; Mode toggle</td><td><code>ai_live</code> flag on the cell view</td><td>cell DAG metadata; consumed by <code>spawn_reactive_engine</code> (pending <code>bd-1bpb</code>)</td></tr>
    <tr><td>quoted prompt body</td><td>prompt turn text</td><td><code>AcpAgentBackend::run</code> draining the agent stream (acp_backend.rs)</td></tr>
    <tr><td><code>&#9656; in&#8594;out tok</code> + Usage meter</td><td><code>AiUsage { input, output }</code></td><td><code>AiRunOutput</code> (ai/mod.rs)</td></tr>
    <tr><td><code>cached</code> chip</td><td>input-hash cache hit</td><td>blake3(prompt + port versions) in <code>NotebookCellRunner</code> (cell_runner.rs)</td></tr>
    <tr><td>context-port <code>&#8595;</code> tokens</td><td>rendered Arrow context</td><td><code>render_port_context</code> / <code>PortContext</code> (ai/context.rs)</td></tr>
    <tr><td><code>&#9888; needs agent</code> state</td><td><code>AiError::Init</code></td><td><code>NullAiBackend</code> (ai/null_backend.rs)</td></tr>
  </tbody>
</table>
</div>
""")


UI mark,Drives / reads,Where it lives
✦ agent tag & Agent selector,AgentConfig resolution,ai_backend_from_config · build_agent_connection (run_context.rs)
manual/LIVE pill & Mode toggle,ai_live flag on the cell view,cell DAG metadata; consumed by spawn_reactive_engine (pending bd-1bpb)
quoted prompt body,prompt turn text,AcpAgentBackend::run draining the agent stream (acp_backend.rs)
▸ in→out tok + Usage meter,"AiUsage { input, output }",AiRunOutput (ai/mod.rs)
cached chip,input-hash cache hit,blake3(prompt + port versions) in NotebookCellRunner (cell_runner.rs)
context-port ↓ tokens,rendered Arrow context,render_port_context / PortContext (ai/context.rs)
⚠ needs agent state,AiError::Init,NullAiBackend (ai/null_backend.rs)


---

# The AI node in **pure cell view** (`viewMode: "cells"`)

Not the DAG graph — the normal linear notebook. An AI node is just a cell: it keeps the notebook cell skeleton from `NotebookCells.tsx` (left `[n]` execution gutter, `<hr>` separators, input → output, right aside) but swaps in four AI-specific things — a violet `✦` run marker, an agent/mode header, a **prompt** body instead of code, and an output that is **generated text** with a token-usage + produced-port footer. Rendered below.

In [4]:
# open-design artifact — AI node in pure cell view
from IPython.display import HTML

HTML(r"""
<div id="cv-root">
<style>
#cv-root{--fg:#111827;--muted:#6b7280;--faint:#9ca3af;--border:#e5e7eb;--border2:#d1d5db;
--ai:#7c3aed;--ai-bg:#f5f3ff;--ai-bd:#ddd6fe;--ok:#10b981;--warn:#f59e0b;--err:#ef4444;
--sans:-apple-system,BlinkMacSystemFont,'Inter','Segoe UI',system-ui,sans-serif;
--mono:'JetBrains Mono','IBM Plex Mono',ui-monospace,Menlo,monospace;
background:#f8fafc;color:var(--fg);font-family:var(--sans);font-size:14px;line-height:1.5;
padding:26px 28px 38px;border-radius:8px;}
#cv-root *{box-sizing:border-box;}
#cv-root h1{font-size:20px;font-weight:700;letter-spacing:-.02em;margin:0;}
#cv-root .sub{color:var(--muted);font-size:13px;margin-top:4px;max-width:720px;}
#cv-root .kicker{font-family:var(--mono);font-size:10.5px;font-weight:600;letter-spacing:.08em;
text-transform:uppercase;color:var(--faint);margin:30px 0 0;border-top:1px solid var(--border);padding-top:13px;}
/* notebook surface — the linear cell column */
#cv-root .nb{background:#fff;border:1px solid var(--border);border-radius:8px;margin-top:14px;
padding:4px 0;max-width:840px;overflow:hidden;}
#cv-root .cell{position:relative;padding:10px 18px 12px 64px;}
#cv-root .hr{border:0;border-top:1px solid var(--border);margin:0;}
/* left execution gutter — mirrors CellExecutionMarker (w-57px, mono 10.5px) */
#cv-root .gutter{position:absolute;left:0;top:14px;width:57px;display:flex;justify-content:center;
font-family:var(--mono);font-size:10.5px;line-height:1.25;color:var(--faint);}
#cv-root .gutter.ai{color:var(--ai);font-weight:600;}
#cv-root .gutter .pulse{display:inline-flex;align-items:center;gap:3px;}
#cv-root .gutter .dot{width:6px;height:6px;border-radius:50%;background:var(--ai);}
/* code input */
#cv-root .code{font-family:var(--mono);font-size:12.5px;color:#1f2937;white-space:pre;overflow-x:auto;}
#cv-root .k{color:#7c3aed;} #cv-root .s{color:#047857;} #cv-root .c{color:var(--faint);}
#cv-root .out{font-family:var(--mono);font-size:12px;color:#374151;margin-top:8px;}
#cv-root .out .muted{color:var(--faint);}
/* AI cell header */
#cv-root .aihead{display:flex;align-items:center;gap:7px;margin-bottom:7px;flex-wrap:wrap;}
#cv-root .tag-ai{display:inline-flex;align-items:center;gap:3px;font-family:var(--mono);font-size:10px;
font-weight:600;color:var(--ai);background:var(--ai-bg);border:1px solid var(--ai-bd);border-radius:4px;padding:1px 6px;}
#cv-root .agent{font-family:var(--mono);font-size:10px;color:var(--muted);}
#cv-root .pill{font-family:var(--mono);font-size:9.5px;border-radius:4px;padding:1px 6px;border:1px solid var(--border2);color:var(--muted);background:#fff;}
#cv-root .pill.live{color:#fff;background:var(--ai);border-color:var(--ai);}
#cv-root .spacer{flex:1;}
#cv-root .runbtn{font-family:var(--sans);font-size:11px;font-weight:500;color:#fff;background:var(--ai);
border:0;border-radius:5px;padding:3px 10px;display:inline-flex;align-items:center;gap:4px;}
#cv-root .runbtn.ghost{color:var(--muted);background:#fff;border:1px solid var(--border2);}
/* prompt input — roman text, violet left rail (signals: language, not code) */
#cv-root .prompt{border-left:2px solid var(--ai-bd);padding:2px 0 2px 12px;font-size:13.5px;color:#374151;line-height:1.5;}
/* AI output — generated prose */
#cv-root .aiout{margin-top:2px;font-size:13.5px;color:#1f2937;line-height:1.6;}
#cv-root .aiout ul{margin:4px 0 0;padding-left:18px;} #cv-root .aiout li{margin:3px 0;}
#cv-root .cursor{display:inline-block;width:7px;height:15px;background:var(--ai);vertical-align:-2px;margin-left:1px;animation:cvb 1s steps(2) infinite;}
@keyframes cvb{50%{opacity:0;}}
/* AI footer — usage / cached / produced port */
#cv-root .aifoot{margin-top:10px;display:flex;align-items:center;gap:9px;flex-wrap:wrap;
font-family:var(--mono);font-size:10px;color:var(--faint);}
#cv-root .chip{font-family:var(--mono);font-size:9.5px;border-radius:4px;padding:0 6px;border:1px solid var(--border);color:var(--muted);background:#fafafa;}
#cv-root .chip.warn{color:#92400e;background:#fffbeb;border-color:#fde68a;}
#cv-root .chip.err{color:#b91c1c;background:#fef2f2;border-color:#fecaca;}
#cv-root .port{display:inline-flex;align-items:center;gap:4px;color:var(--muted);}
#cv-root .gT{font-family:var(--mono);font-size:8px;border:1px solid var(--border2);border-radius:2px;padding:0 2px;color:var(--muted);}
/* right aside hint (cell-type toggle / timing) */
#cv-root .aside{position:absolute;right:10px;top:12px;display:flex;align-items:center;gap:7px;color:var(--faint);}
#cv-root .ico{width:16px;height:16px;border:1px solid var(--border);border-radius:4px;display:inline-flex;align-items:center;justify-content:center;font-size:9px;font-family:var(--mono);}
#cv-root .timing{font-size:11px;color:var(--faint);}
#cv-root .timing .ok{color:var(--ok);} #cv-root .timing .bad{color:var(--err);}
/* anatomy callouts */
#cv-root .co{display:flex;gap:9px;align-items:flex-start;font-size:12.5px;margin-top:9px;}
#cv-root .co b{font-family:var(--mono);font-size:10px;color:#fff;background:var(--ai);border-radius:50%;width:17px;height:17px;
display:inline-flex;align-items:center;justify-content:center;flex:0 0 auto;margin-top:1px;}
#cv-root .co code{font-family:var(--mono);font-size:11px;background:#f1f5f9;border-radius:3px;padding:0 4px;color:#334155;}
#cv-root .row{display:flex;gap:26px;align-items:flex-start;flex-wrap:wrap;}
#cv-root .gallery{display:grid;grid-template-columns:repeat(auto-fill,minmax(330px,1fr));gap:16px;margin-top:14px;}
#cv-root .gcap{font-family:var(--mono);font-size:10.5px;font-weight:600;color:var(--fg);margin:9px 0 1px;}
#cv-root .gcd{font-size:11.5px;color:var(--muted);line-height:1.4;}
#cv-root .mini{background:#fff;border:1px solid var(--border);border-radius:7px;padding:10px 14px 12px 56px;position:relative;}
#cv-root .mini .gutter{top:12px;width:50px;}
</style>

<h1>An AI node is a notebook cell</h1>
<div class="sub">Same skeleton as every other cell — left <code style="font-family:var(--mono);font-size:12px">[n]</code> run gutter, input on top, output below an <code style="font-family:var(--mono);font-size:12px">&lt;hr&gt;</code>, type toggle in the right aside. Four things change for a <code style="font-family:var(--mono);font-size:12px">spur</code> cell: a violet <b>&#10022;</b> run marker, an agent/mode header, the body is a <b>prompt</b> not code, and the output is <b>generated text</b> with a usage + produced-port footer.</div>

<div class="kicker">1 &middot; In the cell stream</div>
<div class="nb">

  <!-- code cell 1 -->
  <div class="cell">
    <div class="gutter">[1]</div>
    <div class="aside"><span class="ico" title="code cell">&lt;/&gt;</span><span class="timing"><span class="ok">&#10003;</span> 0.21 s</span></div>
    <div class="code"><span class="k">import</span> polars <span class="k">as</span> pl
sales = pl.read_parquet(<span class="s">"weekly.csv"</span>)</div>
    <div class="out"><span class="muted">shape: (1_240, 6) &nbsp;&middot;&nbsp; produces </span>sales <span class="muted">v3</span></div>
  </div>
  <hr class="hr"/>

  <!-- code cell 2 -->
  <div class="cell">
    <div class="gutter">[2]</div>
    <div class="aside"><span class="ico" title="code cell">&lt;/&gt;</span><span class="timing"><span class="ok">&#10003;</span> 0.05 s</span></div>
    <div class="code">targets = load_targets()  <span class="c"># quota.json</span></div>
    <div class="out"><span class="muted">produces </span>targets <span class="muted">v1</span></div>
  </div>
  <hr class="hr"/>

  <!-- AI cell -->
  <div class="cell">
    <div class="gutter ai">&#10022;[3]</div>
    <div class="aside"><span class="ico" title="cell type">&#10022;</span><span class="timing"><span class="ok">&#10003;</span> 2.4 s</span></div>
    <div class="aihead">
      <span class="tag-ai">&#10022; AI</span>
      <span class="agent">claude-opus</span>
      <span class="pill">manual</span>
      <span class="spacer"></span>
      <button class="runbtn">&#9654; Run</button>
    </div>
    <div class="prompt">Summarise this week&#8217;s sales against targets in 3 bullets; call out the biggest miss.</div>
    <div class="out" style="margin-top:10px;"><span class="muted" style="font-family:var(--mono);font-size:10px;">output &middot; summary</span></div>
    <div class="aiout">
      <ul>
        <li>Revenue landed at <b>$128k</b>, 14% under the $148k target &mdash; the week&#8217;s biggest miss.</li>
        <li>Enterprise held: 6 of 7 quota reps cleared 90%+ of plan.</li>
        <li>SMB self-serve slipped to 71% of plan; the sub-$50 tier is the drag.</li>
      </ul>
    </div>
    <div class="aifoot">
      <span>&#9656; 1,200 in &#8594; 380 out</span>
      <span class="chip">cached</span>
      <span class="port">produces <span class="gT">T</span> summary <span style="color:var(--faint)">v5</span></span>
    </div>
  </div>
  <hr class="hr"/>

  <!-- code cell 4 -->
  <div class="cell">
    <div class="gutter">[4]</div>
    <div class="aside"><span class="ico" title="code cell">&lt;/&gt;</span><span class="timing"><span class="ok">&#10003;</span> 0.03 s</span></div>
    <div class="code">render_md(summary)  <span class="c"># consumes the text port</span></div>
    <div class="out"><span class="muted">&#10551; rendered markdown report</span></div>
  </div>

</div>

<div class="kicker">2 &middot; Anatomy of the AI cell</div>
<div class="row">
  <div style="flex:0 0 auto;max-width:430px;width:100%;">
    <div class="mini" style="margin-top:14px;">
      <div class="gutter ai">&#10022;[3]</div>
      <div class="aihead">
        <span class="tag-ai">&#10022; AI</span><span class="agent">claude-opus</span><span class="pill live">&#9679; LIVE</span>
        <span class="spacer"></span><button class="runbtn">&#9654; Run</button>
      </div>
      <div class="prompt">Summarise this week&#8217;s sales against targets in 3 bullets; call out the biggest miss.</div>
      <div class="aiout" style="margin-top:9px;font-size:12.5px;"><ul><li>Revenue $128k, 14% under target &mdash; biggest miss.</li><li>Enterprise held; SMB self-serve slipped.</li></ul></div>
      <div class="aifoot"><span>&#9656; 1,200 &#8594; 380</span><span class="chip">cached</span><span class="port">&#10551; <span class="gT">T</span> summary</span></div>
    </div>
  </div>
  <div style="flex:1;min-width:260px;">
    <div class="co"><b>1</b><div>Violet <code>&#10022;[n]</code> run marker replaces the plain <code>[n]</code> gutter &mdash; same `CellExecutionMarker` slot, AI-tinted. <code>&#10022;[*]</code> while the turn streams.</div></div>
    <div class="co"><b>2</b><div>Header row: <code>&#10022; AI</code> + the bound agent (<code>ai_backend_from_config</code>) + a <code>manual</code>/<code>&#9679; LIVE</code> mode pill (<code>ai_live</code>).</div></div>
    <div class="co"><b>3</b><div>Body is a <b>prompt</b> &mdash; roman text on a violet rail, not a syntax-highlighted code editor. It is the cell source, sent to <code>AcpAgentBackend</code>.</div></div>
    <div class="co"><b>4</b><div>Output is <b>generated text</b> (the produced text port&#8217;s content), rendered like a markdown result rather than a kernel repr.</div></div>
    <div class="co"><b>5</b><div>Footer: <code>AiUsage</code> tokens, the <code>cached</code> chip (input-hash hit), and the produced <code>&#10551; T summary</code> port that downstream code cells consume.</div></div>
  </div>
</div>

<div class="kicker">3 &middot; The AI cell across states</div>
<div class="gallery">

  <div>
    <div class="mini">
      <div class="gutter ai"><span class="pulse"><span class="dot"></span></span></div>
      <div class="aihead"><span class="tag-ai">&#10022; AI</span><span class="agent">opus</span><span class="pill live">&#9679; LIVE</span></div>
      <div class="prompt" style="font-size:12.5px;">Summarise sales vs targets&#8230;</div>
      <div class="aiout" style="margin-top:8px;font-size:12.5px;">Revenue landed at $128k, 14% under<span class="cursor"></span></div>
    </div>
    <div class="gcap">running &middot; streaming</div><div class="gcd">Gutter shows a pulsing &#10022; dot; tokens stream into the output as the prompt turn drains.</div>
  </div>

  <div>
    <div class="mini">
      <div class="gutter ai">&#10022;[3]</div>
      <div class="aihead"><span class="tag-ai">&#10022; AI</span><span class="agent">opus</span><span class="pill">manual</span></div>
      <div class="prompt" style="font-size:12.5px;">Summarise sales vs targets&#8230;</div>
      <div class="aifoot" style="margin-top:9px;"><span>&#9656; 1,200 &#8594; 380</span><span class="chip">cached</span></div>
    </div>
    <div class="gcap">cached</div><div class="gcd">Input hash (prompt + consumed port versions) unchanged &mdash; replayed from cache, no agent call, no spend.</div>
  </div>

  <div>
    <div class="mini" style="border-left:2px solid var(--warn);">
      <div class="gutter ai">&#10022;[3]</div>
      <div class="aihead"><span class="tag-ai">&#10022; AI</span><span class="agent">opus</span><span class="pill">manual</span><span class="chip warn">inputs changed</span></div>
      <div class="prompt" style="font-size:12.5px;">Summarise sales vs targets&#8230;</div>
      <div class="aifoot" style="margin-top:9px;"><span style="color:#b45309;">targets v1 &#8594; v2 &middot; re-run to refresh</span></div>
    </div>
    <div class="gcap">stale</div><div class="gcd">A consumed port advanced past the version this cell last ran on; output may be wrong.</div>
  </div>

  <div>
    <div class="mini">
      <div class="gutter ai" style="color:var(--faint);">&#10022;[ ]</div>
      <div class="aihead"><span class="chip warn">&#9888; needs agent</span><span class="spacer"></span><button class="runbtn ghost">&#9654; Run</button></div>
      <div class="prompt" style="font-size:12.5px;color:var(--faint);border-left-color:var(--border2);">Summarise sales vs targets&#8230;</div>
      <div class="aifoot" style="margin-top:9px;"><span style="color:var(--faint);">no agent in .spur/config.toml &middot; run disabled</span></div>
    </div>
    <div class="gcap">needs agent</div><div class="gcd"><code>NullAiBackend</code> &#8594; <code>AiError::Init</code>. Not runnable until an agent is configured.</div>
  </div>

  <div>
    <div class="mini">
      <div class="gutter ai" style="color:var(--err);">&#10022;[3]</div>
      <div class="aihead"><span class="tag-ai">&#10022; AI</span><span class="agent">opus</span><span class="pill">manual</span></div>
      <div class="prompt" style="font-size:12.5px;">Summarise sales vs targets&#8230;</div>
      <div class="aifoot" style="margin-top:9px;"><span class="chip err">&#9888; agent error &middot; timeout</span></div>
    </div>
    <div class="gcap">failed</div><div class="gcd">The prompt turn errored; the gutter marker goes red and the reason shows in the output slot.</div>
  </div>

</div>
</div>
""")
